In [0]:

import requests
import json
from datetime import datetime, timezone

# Open Library API endpoint
API_URL = "https://openlibrary.org/search.json"

# API request parameters
params = {
    "q": "data engineering",
    "limit": 100,
    "fields": "key,title,author_name,first_publish_year,edition_count"
}

# Identify our application to the API
headers = {
    "User-Agent": "BookShelfDataPipeline/1.0 (contact: mkmanpreetkaur0016@gmail.com)"
}

# Databricks Unity Catalog volume location
RAW_PATH = "/Volumes/workspace/default/bookshelf_raw_volume/bronze/open_library"

print("BookShelf Bronze Ingestion Started")


In [0]:

# Make the API request
response = requests.get(
    API_URL,
    params=params,
    headers=headers,
    timeout=30
)

# Check whether the request was successful
response.raise_for_status()

# Parse the response to inspect the records
data = response.json()

# Extract the returned book records
books = data.get("docs", [])

print("API Status:", response.status_code)
print("Total matching works:", data.get("num_found"))
print("Records retrieved:", len(books))

# Display a few records for inspection
for book in books[:5]:
    print(book.get("title"))


In [0]:

# Create the Bronze storage directory
dbutils.fs.mkdirs(RAW_PATH)

# Generate a unique filename using UTC timestamp
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

file_path = f"{RAW_PATH}/books_data_{timestamp}.json"

# Save the original API response
dbutils.fs.put(
    file_path,
    response.text,
    overwrite=False
)

print("Raw data saved successfully!")
print("File location:", file_path)


In [0]:

# List files in the Bronze directory
bronze_files = dbutils.fs.ls(RAW_PATH)

for file in bronze_files:
    print("File:", file.name)
    print("Size:", file.size, "bytes")

# Read the beginning of the saved JSON file
saved_json = dbutils.fs.head(file_path, 1000)

print("\nSaved JSON preview:")
print(saved_json)
